# 📡 Exercise 01 — Fetch OSM Data

**Day 1 · Guided**

Fetch OpenStreetMap amenity data for a district in your city, inspect the GeoDataFrame, extract coordinates, and save locally.

---

### What is OSM?

OpenStreetMap is a free, community-maintained map of the world. No enforced schema — tags are conventions, not standards.

### What is OSMnx?

[OSMnx](https://osmnx.readthedocs.io/) wraps the Overpass API and returns clean GeoDataFrames. Explore data interactively at [overpass-turbo.eu](https://overpass-turbo.eu).

### 1.1 🏘️ Pick your city **and district**

Look up your city in [`docs/cities.md`](../docs/cities.md) and choose a **district / neighbourhood** within it. Fetching a whole city returns too much data and overloads the API.

> Use `"District, City, Country"` — e.g. `"Södermalm, Stockholm, Sweden"`, `"el Poblenou, Barcelona, Spain"`, `"Kreuzberg, Berlin, Germany"`. Verify on [nominatim.openstreetmap.org](https://nominatim.openstreetmap.org/) that the name resolves.

In [2]:
# TODO: Replace with your district + city + country
PLACE = "종로구, Seoul, South Korea"  # e.g. "Södermalm, Stockholm, Sweden"

print(f"Working with: {PLACE}")

Working with: 종로구, Seoul, South Korea


In [3]:
!pip install osmnx

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   --------------- ------------------------ 0.8/2.1 MB 4.6 MB/s eta 0:00:01
   ----------------------------------- ---- 1.8/2.1 MB 4.7 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 4.7 MB/s  0:00:00
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   --- ------------------------------------ 0.8/9.9 MB 5.0 MB/s eta 0:00:02
   ------- -------------------------------- 1.8/9.9 MB 5.1 MB/s eta 0:00:02
   ----------- ---------------------------- 2.9/9.9 MB 4.9 MB/s eta 0:00:02
   --------------- ------------------------ 3.9/9.9 MB 4.9 MB/s eta 0:00:02
   ------------------- -------------------- 4.7/9.9 MB 4.7 MB/s eta 0:00:02
   ----------------------- ---------------- 5.8/9.9 MB 4.6 MB/s eta 0:00:01
   -------------------------- ------------- 6.6/9.9 MB 4.7 MB/s eta 0:00:01
   ------------------------

### 1.2 📥 Fetch amenities

We pass a curated list of amenity types (not *all* amenities) to keep the request small.

In [4]:
import osmnx as ox

AMENITY_TYPES = [
    "restaurant", "cafe", "fast_food", "bar", "pub",
    "pharmacy", "hospital", "clinic", "dentist",
    "school", "university", "library", "kindergarten",
    "bank", "atm", "post_office",
]

print(f"Fetching amenity data for {PLACE}...")
gdf = ox.features_from_place(PLACE, tags={"amenity": AMENITY_TYPES})
gdf = gdf.reset_index()

print(f"Fetched {len(gdf)} features · {len(gdf.columns)} columns")
print(f"Geometry types:\n{gdf.geometry.geom_type.value_counts()}")

Fetching amenity data for 종로구, Seoul, South Korea...
Fetched 1627 features · 172 columns
Geometry types:
Point      1426
Polygon     201
Name: count, dtype: int64


#### Troubleshooting

| Error | Fix |
|-------|-----|
| `InsufficientResponseError` | Check spelling — use `"District, City, Country"` format |
| `ConnectionError` / timeout | Wait 30 s, retry |
| 0 features | Verify name on [nominatim.openstreetmap.org](https://nominatim.openstreetmap.org/) |
| Very slow | Pick a smaller district or reduce `AMENITY_TYPES` |

> ⚠️ The Overpass API is a free shared resource. Don't spam requests — save locally and reload.

### 1.3 🔎 Inspect the raw data

In [5]:
# Look at one feature — all its columns and values
gdf.iloc[0]

element                                   node
id                                   368622407
geometry        POINT (126.9538429 37.6033425)
alt_name:ko                       상명사대부속여자고등학교
amenity                                 school
                             ...              
name:ko-Hani                               NaN
name:zh-Hant                               NaN
dispensing                                 NaN
surface                                    NaN
type                                       NaN
Name: 0, Length: 172, dtype: object

In [6]:
# TODO: Look at a few more features — are they all structured the same way?
# Try: gdf.iloc[1], gdf.iloc[-1]
# What columns are always present? Which ones are mostly NaN?

In [7]:
gdf.iloc[-1]

element                                                       way
id                                                     1492219951
geometry        POLYGON ((126.9578684 37.5757181, 126.9578514 ...
alt_name:ko                                                   NaN
amenity                                                      bank
                                      ...                        
name:ko-Hani                                                  NaN
name:zh-Hant                                                  NaN
dispensing                                                    NaN
surface                                                       NaN
type                                                          NaN
Name: 1626, Length: 172, dtype: object

**Think:** Which columns does every feature have? Why are most columns mostly NaN?

### 1.4 🌐 Extract coordinates → DataFrame

We need plain `lat`/`lon` columns. `geometry.centroid` works for both Points and Polygons.

In [8]:
import pandas as pd

# Extract lat/lon from geometry centroids (works for Points and Polygons)
gdf["lat"] = gdf.geometry.centroid.y
gdf["lon"] = gdf.geometry.centroid.x

# Convert to regular DataFrame (drop geometry column)
df = pd.DataFrame(gdf.drop(columns=["geometry"]))

# Rename osmid → id for consistency
if "osmid" in df.columns:
    df = df.rename(columns={"osmid": "id"})

# Drop osmnx metadata columns
for col in ["nodes", "ways"]:
    if col in df.columns:
        df = df.drop(columns=[col])

print(f"Shape: {df.shape}  (rows, columns)")
print(f"Columns: {len(df.columns)}")
df.head(10)

Shape: (1627, 173)  (rows, columns)
Columns: 173


C:\Users\maria\AppData\Local\Temp\ipykernel_2776\717398241.py:4: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["lat"] = gdf.geometry.centroid.y
C:\Users\maria\AppData\Local\Temp\ipykernel_2776\717398241.py:5: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["lon"] = gdf.geometry.centroid.x


,element,id,alt_name:ko,amenity,isced:level,name,name:en,name:ko,name:ko-Latn,name:ja,...,brand:ja,surveillance,emergency,name:ko-Hani,name:zh-Hant,dispensing,surface,type,lat,lon
0,node,368622407,상명사대부속여자고등학교,school,3,상명대학교 사범대학 부속여자고등학교,Sangmyeong University Busok Womens High School,상명대학교 사범대학 부속여자고등학교,Sangmyeongdaehakgyo Sabeomdaehak Busokyeojagod...,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37.603342,126.953843
1,node,368622883,NaN,school,3,서울정보산업고등학교,Seoul Information Industry High School,서울정보산업고등학교,Seouljeongbosaneopgodeunghakgyo,ソウル情報産業高等学校,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37.578086,127.013902
2,node,368623187,NaN,school,2;3,대신중고등학교,Daesinjung High School,대신중고등학교,Daesinjunggodeunghakgyo,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37.574339,126.962573
3,node,368638884,NaN,bank,NaN,동서울농협,Dong Seoul Nonghyup,동서울농협,Dongseoullonghyeop,東ソウル農協,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37.572375,127.014181
4,node,368650195,NaN,university,NaN,상명대학교 서울캠퍼스,Sangmyeong University Seoul Campus,상명대학교 서울캠퍼스,Sangmyeongdaehakgyo Seoulkaempeoseu,祥明大学校 ソウルキャンパス,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37.601739,126.954885
5,node,368652190,NaN,university,NaN,서울장신대학교,Seoul Jangsin University,서울장신대학교,Seouljangsindaehakgyo,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37.574697,127.022472
6,node,368652571,NaN,library,NaN,사회과학도서관,Social Science Library,사회과학도서관,Sahoegwahakdoseogwan,社会科学図書館,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37.573671,126.964473
7,node,368652920,NaN,library,NaN,석천문화관문고,Seokcheonmunhwagwan Library,석천문화관문고,Seokcheonmunhwagwanmungo,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37.590259,126.997752
8,node,368652922,NaN,library,NaN,쌍용아파트2단지문고,Ssangyong Apt. 2 Complex Library,쌍용아파트2단지문고,Ssangyongapateu 2 Danjimungo,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37.580790,127.011783
9,node,368654150,NaN,library,NaN,숭인1동문고,Sungin 1 Dong Library,숭인1동문고,Sungin 1 Dongmungo,崇仁1洞文庫,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37.577889,127.015614


In [9]:
type(gdf)

geopandas.geodataframe.GeoDataFrame

In [10]:
# OSMnx already expanded tags into separate columns — let's see what we have
meta_cols = {"element_type", "id", "lat", "lon"}
tag_cols = sorted([c for c in df.columns if c not in meta_cols])
print(f"Number of tag columns: {len(tag_cols)}")

# TODO: Print the first 20 tag column names
# Hint: tag_cols[:20]

Number of tag columns: 170


In [11]:
# Verify key columns are present
key_cols = ["id", "lat", "lon", "amenity", "name", "cuisine", "opening_hours"]
for col in key_cols:
    present = col in df.columns
    filled = df[col].notna().sum() if present else 0
    print(f"  {col}: {'✓' if present else '✗'}  ({filled} non-null)")

print(f"\nTotal shape: {df.shape}")

  id: ✓  (1627 non-null)
  lat: ✓  (1627 non-null)
  lon: ✓  (1627 non-null)
  amenity: ✓  (1627 non-null)
  name: ✓  (1441 non-null)
  cuisine: ✓  (418 non-null)
  opening_hours: ✓  (102 non-null)

Total shape: (1627, 173)


In [12]:
d = {"a" : 1, "b" : 2}
d["a"]

1

In [13]:
gdf["amenity"].unique()

<StringArray>
[      'school',         'bank',   'university',      'library',
      'dentist',  'post_office', 'kindergarten',     'hospital',
   'restaurant',         'cafe',          'bar',          'atm',
    'fast_food',     'pharmacy',          'pub',       'clinic']
Length: 16, dtype: str

In [14]:
gdf["amenity"].value_counts()

amenity
restaurant      900
cafe            394
bank             57
fast_food        53
bar              50
school           42
library          25
pharmacy         24
post_office      20
pub              19
university       11
hospital          9
dentist           8
atm               8
kindergarten      4
clinic            3
Name: count, dtype: int64

In [15]:
gdf["geometry"][58].centroid.x

126.9540974

### 1.5 💾 Save locally

In [16]:
import os
os.makedirs("../data", exist_ok=True)

# Derive a slug from the place name for file naming
city_slug = PLACE.split(",")[0].strip().lower().replace(" ", "_")
gdf.to_file(f"../data/raw_{city_slug}.gpkg", driver="GPKG")

print(f"Saved to ../data/raw_{city_slug}.gpkg")
print(f"Size: {os.path.getsize(f'../data/raw_{city_slug}.gpkg') / 1024:.0f} KB")

Saved to ../data/raw_종로구.gpkg
Size: 740 KB


### 1.6 👀 First look

In [ ]:
# TODO: Run these commands and study the output

# 1. What are the dtypes?
# df.dtypes

# 2. How much data is missing per column?
# df.info()

# 3. What are the most common amenity types in your city?
# df["amenity"].value_counts().head(10)

In [34]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1627 entries, 0 to 1626
Columns: 173 entries, element to lon
dtypes: float64(2), int64(1), object(170)
memory usage: 2.1+ MB


In [35]:
df["amenity"].value_counts().head(10)

,count
amenity,
restaurant,900
cafe,394
bank,57
fast_food,53
bar,50
school,42
library,25
pharmacy,24
post_office,20


---

✅ Raw GeoPackage in `data/` · ✅ Flat DataFrame with lat/lon · ✅ First look at the data

**Next →** [Exercise 02 — Explore & Clean](02_explore_and_clean.ipynb)